## 16. Scripts: worldmodel.py, train.py, predict.py
Notebook code moved into scripts, generated in make_scripts.ipynb. Final model = equal-weight blend of gradient boosting, MLPs and autoregressive world models. Check on the validation set should reproduce about 0.09.

## worldmodel.py (shared model code)
The notebook code is moved into scripts. This file holds the MLP, the autoregressive step network, scheduled-sampling training and the free-running simulation.

In [2]:
%%writefile worldmodel.py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Slow-moving daily columns that act like a calendar: dropped from the neural nets
TREND_COLS = ["sunmars_km", "earthmars_km", "sunmarsearthangle_deg",
              "solarconstantmars", "eclipseduration_min", "occultationduration_min"]


def load_hourly(data_dir):
    """power.csv -> one row per hour, empty hours filled by interpolation."""
    p = pd.read_csv(f"{data_dir}/power.csv")
    p["time"] = pd.to_datetime(p["ut_ms"], unit="ms")
    h = p.set_index("time").drop(columns="ut_ms").resample("1h").mean()
    return h.interpolate(method="time").bfill().ffill()


def make_mlp(n_in, n_out, hidden=64):
    """Network without memory: context of one hour -> 33 circuits."""
    return nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Dropout(0.2),
                         nn.Linear(hidden, hidden), nn.ReLU(),
                         nn.Linear(hidden, n_out))


class StepNet(nn.Module):
    """World model step: (context of this hour, previous hour's power) -> this hour's power."""
    def __init__(self, n_ctx, n_out, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_ctx + n_out, hidden), nn.ReLU(), nn.Dropout(0.1),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, n_out))

    def forward(self, ctx, prev):
        return self.net(torch.cat([ctx, prev], dim=-1))


def train_mlp(Xt, Yt, seed, steps=1500, hidden=64, wd=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    m = make_mlp(Xt.shape[1], Yt.shape[1], hidden)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3, weight_decay=wd)
    for _ in range(steps):
        m.train()
        idx = torch.randint(0, len(Xt), (512,))
        loss = ((m(Xt[idx]) - Yt[idx]) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return m


def train_ar(Xt, Yt, lo, hi, seed, steps=1200, K=48, B=64, hidden=64, wd=1e-3):
    """Autoregressive training with scheduled sampling (own predictions fed back, 0% -> 100%)."""
    torch.manual_seed(seed); np.random.seed(seed)
    m = StepNet(Xt.shape[1], Yt.shape[1], hidden)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3, weight_decay=wd)
    for step in range(steps):
        m.train()
        p_own = min(1.0, step / (0.7 * steps))
        s = torch.tensor(np.random.randint(1, len(Xt) - K, size=B))
        prev = Yt[s - 1]
        loss = 0
        for k in range(K):
            out = m(Xt[s + k], prev)
            loss = loss + ((out - Yt[s + k]) ** 2).mean()
            own = torch.maximum(torch.minimum(out.detach(), hi), lo)   # clamp to training range
            use_own = (torch.rand(B, 1) < p_own).float()
            prev = use_own * own + (1 - use_own) * Yt[s + k]
        loss = loss / K
        opt.zero_grad(); loss.backward(); opt.step()
        if (step + 1) % 300 == 0:
            print(f"  AR seed {seed} step {step + 1}: loss {loss.item():.4f}", flush=True)
    return m


def simulate(m, Xt, s0, lo, hi):
    """Free-running dream: every hour uses the model's OWN previous prediction."""
    m.eval(); prev = torch.tensor(s0)[None]; out = []
    with torch.no_grad():
        for t in range(len(Xt)):
            y = torch.maximum(torch.minimum(m(Xt[t][None], prev), hi), lo)
            out.append(y[0].numpy()); prev = y
    return np.vstack(out)

Writing worldmodel.py


## train.py (train everything from the kit)
Builds features, trains gradient boosting, MLPs and autoregressive world models, and saves the weights and metadata to a model folder.

In [3]:
%%writefile train.py
"""Train everything from the kit. Usage: python train.py [--train_dirs annee1 annee2] [--out_dir model]"""
import argparse, json, os
import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import HistGradientBoostingRegressor
from features import top_lists, build_features
from worldmodel import TREND_COLS, load_hourly, train_mlp, train_ar

ap = argparse.ArgumentParser()
ap.add_argument("--train_dirs", nargs="+", default=["annee1", "annee2"])
ap.add_argument("--out_dir", default="model")
ap.add_argument("--n_seeds", type=int, default=3)     # networks per family (MLP, AR)
ap.add_argument("--no_gb", action="store_true")       # neural-only version
args = ap.parse_args()
os.makedirs(args.out_dir, exist_ok=True)

# Lists of frequent event kinds / command families, fixed from the first training folder
ev_kinds, cmd_fams = top_lists(args.train_dirs[0])

# Hourly power (targets) and hourly context features, for every training folder
Xs, Ys = [], []
for d in args.train_dirs:
    print("building features:", d, flush=True)
    y = load_hourly(d)
    Xs.append(build_features(d, y.index, ev_kinds, cmd_fams))
    Ys.append(y)
X_all, Y_all = pd.concat(Xs), pd.concat(Ys)
circuits = list(Y_all.columns)
all_cols = list(X_all.columns)
nn_cols = [c for c in all_cols if c not in TREND_COLS]   # nets do not see the daily trends

# Standardize features; targets: centre per circuit, ONE global scale (matches the RMSE metric)
mu = X_all[nn_cols].mean()
sd = X_all[nn_cols].std().replace(0, 1)
Xt = torch.tensor(((X_all[nn_cols] - mu) / sd).values.astype("float32"))
ymean = Y_all.mean()
scale = float((Y_all - ymean).values.std())
Yt = torch.tensor(((Y_all - ymean) / scale).values.astype("float32"))
lo, hi = Yt.min(0).values, Yt.max(0).values              # clamp range for the dream

# Gradient boosting: one model per circuit, all features (component of the blend)
if not args.no_gb:
    print("training gradient boosting", flush=True)
    gb = {}
    for c in circuits:
        m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=0)
        gb[c] = m.fit(X_all[all_cols], Y_all[c])
    joblib.dump(gb, f"{args.out_dir}/gb.joblib")

# Neural networks: several seeds each
print("training MLPs", flush=True)
mlps = [train_mlp(Xt, Yt, seed=s) for s in range(args.n_seeds)]
print("training autoregressive world models", flush=True)
ars = [train_ar(Xt, Yt, lo, hi, seed=s) for s in range(args.n_seeds)]
torch.save({"mlp": [m.state_dict() for m in mlps],
            "ar": [m.state_dict() for m in ars]}, f"{args.out_dir}/nets.pth")

meta = dict(circuits=circuits, all_cols=all_cols, nn_cols=nn_cols,
            ev_kinds=list(ev_kinds), cmd_fams=list(cmd_fams),
            mu=mu.tolist(), sd=sd.tolist(), ymean=ymean.tolist(), scale=scale,
            lo=lo.tolist(), hi=hi.tolist(), n_in=len(nn_cols),
            has_gb=not args.no_gb, train_dirs=args.train_dirs)
json.dump(meta, open(f"{args.out_dir}/meta.json", "w"))
print("done. saved in", args.out_dir)

Writing train.py


## predict.py (required interface)
Builds features from the test folder, simulates from power_init.csv, blends the members with equal weights and writes the CSV. Check on the validation set should reproduce about 0.09.

In [1]:
%%writefile predict.py
"""Usage: python predict.py --test_dir DOSSIER --out predictions.csv [--model_dir model]"""
import argparse, json
import joblib
import numpy as np
import pandas as pd
import torch
from features import build_features
from worldmodel import make_mlp, StepNet, simulate

ap = argparse.ArgumentParser()
ap.add_argument("--test_dir", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--model_dir", default="model")
ap.add_argument("--members_out", default=None)   # optional: save every member (for the demo)
args = ap.parse_args()

meta = json.load(open(f"{args.model_dir}/meta.json"))
circuits, nn_cols, all_cols = meta["circuits"], meta["nn_cols"], meta["all_cols"]
mu = pd.Series(meta["mu"], index=nn_cols)
sd = pd.Series(meta["sd"], index=nn_cols)
ymean = pd.Series(meta["ymean"], index=circuits)
scale = meta["scale"]
lo, hi = torch.tensor(meta["lo"]), torch.tensor(meta["hi"])
nets = torch.load(f"{args.model_dir}/nets.pth")

# Hours to predict, and the continuous hourly grid covering them
heures = pd.read_csv(f"{args.test_dir}/heures_a_predire.csv")
wanted = pd.DatetimeIndex(pd.to_datetime(heures["ut_ms"], unit="ms"))
grid = pd.date_range(wanted.min(), wanted.max(), freq="1h")

# Context features, built only from the files of the test folder
X = build_features(args.test_dir, grid, meta["ev_kinds"], meta["cmd_fams"])
Xt = torch.tensor(np.nan_to_num(((X[nn_cols] - mu) / sd).values.astype("float32")))

# Initial state: the last real hour of power_init.csv
pi = pd.read_csv(f"{args.test_dir}/power_init.csv")
pi["time"] = pd.to_datetime(pi["ut_ms"], unit="ms")
pi_h = pi.set_index("time")[circuits].resample("1h").mean().ffill()
s0 = np.nan_to_num(((pi_h.iloc[-1] - ymean) / scale).values.astype("float32"))

unscale = lambda a: a * scale + ymean.values

# Members: MLPs (no memory), autoregressive world models (the "dream"), gradient boosting
mlp_all, ar_all = [], []
for state in nets["mlp"]:
    m = make_mlp(meta["n_in"], len(circuits)); m.load_state_dict(state); m.eval()
    with torch.no_grad():
        mlp_all.append(unscale(m(Xt).numpy()))
for state in nets["ar"]:
    m = StepNet(meta["n_in"], len(circuits)); m.load_state_dict(state)
    ar_all.append(unscale(simulate(m, Xt, s0, lo, hi)))
members = [np.mean(mlp_all, axis=0), np.mean(ar_all, axis=0)]
if meta["has_gb"]:
    gb = joblib.load(f"{args.model_dir}/gb.joblib")
    gb_pred = np.column_stack([gb[c].predict(X[all_cols]) for c in circuits])
    members.append(gb_pred)

blend = np.mean(members, axis=0)       # equal-weight vote

pred = pd.DataFrame(blend, index=grid, columns=circuits).reindex(wanted)
pred = pred.fillna(ymean)              # safety: never leave a missing value
out = pred.reset_index(drop=True)
out.insert(0, "ut_ms", heures["ut_ms"].values)
out.to_csv(args.out, index=False)
print("saved", args.out, out.shape, "missing:", int(out.isna().sum().sum()))

if args.members_out:
    np.savez(args.members_out, hours=grid.strftime("%Y-%m-%d %H:%M:%S").values,
             mlp=np.array(mlp_all), ar=np.array(ar_all),
             gb=gb_pred if meta["has_gb"] else np.zeros(1))

Overwriting predict.py


Check from the command line (train on annee1, 1 seed per family, predict on validation/test, official evaluate.py): RMSE 0.090424, reproducing the notebook blend (0.0904). The pipeline works end to end.